# Bài 2 — Titanic, Câu 2

Yêu cầu lập trình Titanic trang PDF 15–16

**Sinh viên:** NGUYỄN LỘC — **MSSV:** 6451071043

**Lưu ý:** Notebook tự chạy độc lập sau khi có dataset. Khi dùng Colab, cài `mlxtend`, `openpyxl` và các thư viện nếu cần.

In [ ]:
!pip -q install mlxtend openpyxl seaborn scikit-learn beautifulsoup4

In [ ]:
from pathlib import Path
import os, re, io, zipfile, time, warnings, string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from sklearn.preprocessing import MinMaxScaler
from bs4 import BeautifulSoup
warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid')
HO_TEN = 'NGUYỄN LỘC'
MSSV = '6451071043'
BASE = Path('/content/lab1_data') if Path('/content').exists() else Path('/mnt/data')
BASE.mkdir(parents=True, exist_ok=True)
OUTPUT = BASE / 'results'; OUTPUT.mkdir(exist_ok=True)
def identity(): print(f'Họ tên: {HO_TEN} | MSSV: {MSSV}')
def pick_file(keywords, extensions=('.csv','.xlsx','.xls','.txt','.html','.htm')):
    fs=[p for p in BASE.rglob('*') if p.is_file() and p.suffix.lower() in extensions and not OUTPUT in p.parents]
    for keys in keywords:
        matches=[f for f in fs if all(w in f.name.lower() for w in keys)]
        if matches:return sorted(matches,key=lambda f:len(f.name))[0]
    raise FileNotFoundError(f'Không tìm thấy tập tin chứa từ khóa {keywords} trong {BASE}. File hiện có: {[x.name for x in fs[:30]]}')
def read_table(file, **kw):
    if file.suffix.lower() in ('.xls','.xlsx'):return pd.read_excel(file,**kw)
    for enc in ['utf-8-sig','utf-8','cp1252','latin-1']:
        try:
            df=pd.read_csv(file,encoding=enc,**kw)
            if df.shape[1]==1 and ';' in str(df.columns[0]):df=pd.read_csv(file,encoding=enc,sep=';',**kw)
            return df
        except UnicodeDecodeError:continue
    return pd.read_csv(file,encoding='latin-1',**kw)
def norm_cols(df):
    df=df.copy(); df.columns=[re.sub(r'[^a-z0-9]+','',str(c).strip().lower()) for c in df.columns]
    return df
def rule_table(freq, confidence):
    if freq.empty:return pd.DataFrame()
    try:r=association_rules(freq,metric='confidence',min_threshold=confidence,num_itemsets=int(freq.attrs.get('n_transactions',0)) or None)
    except TypeError:r=association_rules(freq,metric='confidence',min_threshold=confidence)
    if len(r):
        r=r.copy()
        for c in ['antecedents','consequents']:r[c]=r[c].map(lambda s:', '.join(sorted(map(str,s))))
    return r
identity(); print('Thư mục dữ liệu:', BASE)
# Tự dò dữ liệu được Colab upload vào /content hoặc LAB1_data.
def ensure_dataset(filename):
    for d in [BASE,Path('/content'),Path('/content/LAB1_data'),Path('/mnt/data')]:
        p=d/filename
        if p.exists(): return p
    try:
        from google.colab import files
        print('Upload dataset:',filename)
        upl=files.upload()
        for name,b in upl.items(): (BASE/Path(name).name).write_bytes(b)
        if (BASE/filename).exists():return BASE/filename
    except ImportError:pass
    raise FileNotFoundError(filename)


### Chuẩn bị — Đọc Titanic

In [ ]:
titanic_file=ensure_dataset('titanic.csv')
titanic=norm_cols(read_table(titanic_file));titanic=titanic.rename(columns={'survival':'survived','class':'pclass'})

### Kết quả bài làm

In [9]:
identity();print('TOP 5 cột nhiều dữ liệu thiếu');display(titanic.isna().sum().sort_values(ascending=False).head(5).to_frame('Số NA'))

Họ tên:  NGUYỄN LỘC | MSSV: 6451071043
TOP 5 cột nhiều dữ liệu thiếu


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,Số NA
body,1188
cabin,1014
boat,823
homedest,564
age,263


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag